<a href="https://colab.research.google.com/github/harinigayam/Spam-Detection-Using-SVM/blob/main/SMS_Spam_Detection_using_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — Install dependencies
!pip install -q kaggle scikit-learn joblib matplotlib seaborn imbalanced-learn pandas nltk
# Cell 2 — Upload your dataset file
from google.colab import files

print(" Upload your SMS dataset CSV or SMSSpamCollection file")
uploaded = files.upload()  # Choose the file from your computer

for fn in uploaded.keys():
    print("Uploaded:", fn)
# Cell 3 — Load and preprocess the uploaded data
import pandas as pd
import os

# Automatically detect uploaded CSV or text file
candidates = [f for f in os.listdir() if f.endswith('.csv') or f.endswith('.txt')]
if not candidates:
    raise FileNotFoundError("No dataset file found. Upload again and rerun.")
csv_path = candidates[0]
print(" Using dataset file:", csv_path)

# Try different common formats
try:
    df = pd.read_csv(csv_path, encoding='latin-1', usecols=[0,1], names=['label','message'], header=0)
except Exception:
    df = pd.read_csv(csv_path, sep='\t', encoding='latin-1', names=['label','message'], header=None)

# Basic cleanup
df.label = df.label.str.strip()
df = df.dropna().reset_index(drop=True)
print(" Loaded dataset with shape:", df.shape)
df.head()
# Cell 4 — Text preprocessing & train-test split
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# Encode 'spam' → 1, 'ham' → 0
le = LabelEncoder()
df['label_num'] = le.fit_transform(df['label'])

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(df['message'], df['label_num'], test_size=0.2, random_state=42, stratify=df['label_num'])

# TF-IDF vectorization (word + char)
tfidf_word = TfidfVectorizer(ngram_range=(1,2), analyzer='word', max_features=5000)
X_train_tfidf = tfidf_word.fit_transform(X_train)
X_test_tfidf = tfidf_word.transform(X_test)
# Cell 5 — Train the model
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score

svc = LinearSVC()
model = CalibratedClassifierCV(svc)  # adds probability calibration

print(" Training the model...")
model.fit(X_train_tfidf, y_train)
print(" Model training complete!")
# Cell 6 — Evaluation
y_pred = model.predict(X_test_tfidf)

print(" Classification Report:")
print(classification_report(y_test, y_pred, target_names=['ham','spam']))

print(" F1 Score:", f1_score(y_test, y_pred))
print(" ROC-AUC:", roc_auc_score(y_test, y_pred))
# Cell 7 — Save trained model
import joblib, os

os.makedirs("artifacts", exist_ok=True)
joblib.dump(model, "artifacts/spam_model.joblib")
joblib.dump(tfidf_word, "artifacts/tfidf_vectorizer.joblib")

print(" Model & vectorizer saved to /artifacts/")
# Cell 8 — Test custom SMS
sample = ["Congratulations! You've won a free ticket!",
          "Can we meet tomorrow for class?"]

X_sample = tfidf_word.transform(sample)
pred = model.predict(X_sample)
for text, p in zip(sample, pred):
    label = "SPAM " if p == 1 else "HAM "
    print(f"{label} → {text}")
from google.colab import files
files.download('artifacts/spam_model.joblib')
files.download('artifacts/tfidf_vectorizer.joblib')


 Upload your SMS dataset CSV or SMSSpamCollection file


Saving SPAM text message 20170820 - Data (1) (1).csv to SPAM text message 20170820 - Data (1) (1).csv
Uploaded: SPAM text message 20170820 - Data (1) (1).csv
 Using dataset file: SPAM text message 20170820 - Data (1) (1).csv
 Loaded dataset with shape: (5572, 2)
 Training the model...
 Model training complete!
 Classification Report:
              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       966
        spam       0.98      0.91      0.94       149

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.97      1115
weighted avg       0.98      0.98      0.98      1115

 F1 Score: 0.9407665505226481
 ROC-AUC: 0.951467339197132
 Model & vectorizer saved to /artifacts/
SPAM  → Congratulations! You've won a free ticket!
HAM  → Can we meet tomorrow for class?


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>